In [63]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = GoogleGenerativeAI(model="gemini-3.6-flash", google_api_key="API_KEY")

In [64]:
loader = DirectoryLoader(
    "F:\LLM\knowledge_base",
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

docs = loader.load()
print(f"Loaded {len(docs)} markdown files")

Loaded 25 markdown files


In [65]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [66]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [67]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [68]:
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

In [69]:
print(vectorstore._collection.count())

75


In [70]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke("What is the refund policy?")
print(response)

c:\Users\jesse\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, the refund policy includes the following rules:

* **General Refunds:** Customers can request a refund within 7 days of purchase if the service has not been fully utilized.
* **Cancellations:**
  * **More than 48 hours before departure:** Full refund.
  * **Within 48 hours before departure:** 80% refund.
  * **Within 12 hours before departure:** Only taxes and government charges are refundable.
* **Non-refundable Cases:**
  * Promotional bookings
  * Gift cards
  * Loyalty point purchases
  * No-show bookings
* **Processing Time:** Refunds are processed within 5–7 business days.
